In [28]:
%matplotlib inline
import sys
import os
sys.path.append(os.path.abspath(".."))
import importlib

from functions import color_palettes
importlib.reload(color_palettes)
from functions.color_palettes import *

from functions import utils
importlib.reload(utils)
from functions.utils import *

from functions import config
importlib.reload(config)
from functions.config import *

In [ ]:
#I have done some adjustments also directly on the cgm excel, so the cgm_path has the updated one
save_dir='C:\Users\aoezek\Desktop\master_thesis\master_thesis\finalised_clean_notebooks\excels'

In [ ]:
cgm_complete, participant_ids, metadata = load_and_filter_complete_cgm(
    cgm_path=cgm_path,
    metadata_path=metadata_path,
    days_to_keep=days,
)
metadata= clean_string_column(metadata, string_cols)
completed_cgm_metadata = metadata[metadata['complete_cgm_data'] == 'yes']

---
##### FUNDAMENTAL STATS
---

In [ ]:
cgm_complete_long = reshape_cgm_multicolumn(cgm_complete)
statistical_overview=(cgm_complete_long.groupby('Participant_id')['Glucose_Value'].agg(['mean','std','min','max']).reset_index())
#statistical_overview.to_excel(f'{save_dir}\statistical_overview.xlsx', index=False)

---
##### TIME IN RANGE
---

In [ ]:
time_in_range_df=compute_time_in_range(cgm_complete, participant_col = "Participant_id", glucose_col = "Glucose_Value")
#time_in_range_df.to_excel(f'{save_dir}\time_in_range.xlsx', index=False)

---
##### COSINOR FIT (circadian features)
---

In [ ]:
cosinor_df = fit_cosinor(cgm_complete)
cosinor_df.to_excel(f'{save_dir}\cosinor.xlsx', index=False)

---
##### SPEED AND ACCELERATION
---

In [ ]:
speed_acc_df = compute_speed_acceleration(cgm_complete)
speed_acc_df.to_excel(f'{dir}\speed_acc.xlsx', index=False)

---

In [ ]:
# creating a cgm extracted features excel, combining the above dfs
cgm_extracted_features = (
    statistical_overview
    .merge(time_in_range_df, on="Participant_id", how="outer")
    .merge(cosinor_df, on="Participant_id", how="outer")
    .merge(speed_acc_df, on="Participant_id", how="outer")
    .merge(
        completed_cgm_metadata[["ID", "age", "sex"]].rename(columns={"ID": "Participant_id"}),
        on="Participant_id",
        how="left"
    )
)
cgm_extracted_features.to_excel(r'C:\Users\aoezek\Desktop\master_thesis\master_thesis\finalised_clean_notebooks\excels\cgm_extracted_features_46.xlsx')

---
##### Some Exploratory Visuals
---

In [ ]:
plot_cgm_over_days(cgm_complete, participant='AL01')

In [ ]:
plot_one_day_cgm(cgm_complete, "AL01", "2024-12-04")